In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| **Build Model** \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Build the Neural Network
========================

Neural networks comprise of layers/modules that perform operations on
data. The [torch.nn](https://pytorch.org/docs/stable/nn.html) namespace
provides all the building blocks you need to build your own neural
network. Every module in PyTorch subclasses the
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
A neural network is a module itself that consists of other modules
(layers). This nested structure allows for building and managing complex
architectures easily.

In the following sections, we\'ll build a neural network to classify
images in the FashionMNIST dataset.


In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

Get Device for Training
=======================

We want to be able to train our model on an
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


Define the Class
================

We define our neural network by subclassing `nn.Module`, and initialize
the neural network layers in `__init__`. Every `nn.Module` subclass
implements the operations on input data in the `forward` method.


In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

We create an instance of `NeuralNetwork`, and move it to the `device`,
and print its structure.


In [5]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


To use the model, we pass it the input data. This executes the model\'s
`forward`, along with some [background
operations](https://github.com/pytorch/pytorch/blob/270111b7b611d174967ed204776985cefca9c144/torch/nn/modules/module.py#L866).
Do not call `model.forward()` directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0
corresponding to each output of 10 raw predicted values for each class,
and dim=1 corresponding to the individual values of each output. We get
the prediction probabilities by passing it through an instance of the
`nn.Softmax` module.


In [7]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([2])


------------------------------------------------------------------------


Model Layers
============

Let\'s break down the layers in the FashionMNIST model. To illustrate
it, we will take a sample minibatch of 3 images of size 28x28 and see
what happens to it as we pass it through the network.


In [8]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


nn.Flatten
==========

We initialize the
[nn.Flatten](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html)
layer to convert each 2D 28x28 image into a contiguous array of 784
pixel values ( the minibatch dimension (at dim=0) is maintained).


In [9]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


nn.Linear
=========

The [linear
layer](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
is a module that applies a linear transformation on the input using its
stored weights and biases.


In [10]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


nn.ReLU
=======

Non-linear activations are what create the complex mappings between the
model\'s inputs and outputs. They are applied after linear
transformations to introduce *nonlinearity*, helping neural networks
learn a wide variety of phenomena.

In this model, we use
[nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
between our linear layers, but there\'s other activations to introduce
non-linearity in your model.


In [11]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.1828, -0.0801, -0.5296, -0.3462,  0.1082, -0.0797,  0.8296, -0.1277,
          0.0720, -0.2255,  0.4385, -0.4438,  0.1917, -0.0286, -0.0729, -0.3297,
         -0.0852,  0.1502,  0.5959, -0.1617],
        [-0.3209,  0.0156, -0.3777, -0.2495,  0.4353,  0.1426,  0.5280, -0.2657,
          0.0268,  0.0352,  0.3966, -0.2593,  0.6726, -0.2517, -0.0409, -0.5488,
          0.2707,  0.1517,  0.5053, -0.3343],
        [-0.2205, -0.1144, -0.3673, -0.2803,  0.5507, -0.2688,  0.7138,  0.1426,
         -0.1517, -0.0050,  0.5175, -0.3609,  0.2773, -0.2800,  0.2354, -0.5275,
          0.3462, -0.0140,  0.5595, -0.1316]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.1828, 0.0000, 0.0000, 0.0000, 0.1082, 0.0000, 0.8296, 0.0000, 0.0720,
         0.0000, 0.4385, 0.0000, 0.1917, 0.0000, 0.0000, 0.0000, 0.0000, 0.1502,
         0.5959, 0.0000],
        [0.0000, 0.0156, 0.0000, 0.0000, 0.4353, 0.1426, 0.5280, 0.0000, 0.0268,
         0.0352, 0.3966, 0.0000, 0.6726, 0.0000, 0.00

nn.Sequential
=============

[nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
is an ordered container of modules. The data is passed through all the
modules in the same order as defined. You can use sequential containers
to put together a quick network like `seq_modules`.


In [12]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

In [13]:
logits

tensor([[-0.3031,  0.0581, -0.0655, -0.1455,  0.2058,  0.0739, -0.0549, -0.0158,
          0.1589,  0.0985],
        [-0.3939, -0.0554, -0.2136, -0.2324,  0.0745,  0.1500, -0.1638, -0.0875,
          0.2636,  0.0682],
        [-0.3522, -0.0310, -0.1198, -0.1511,  0.0527,  0.0312, -0.1779, -0.1033,
          0.1293,  0.0320]], grad_fn=<AddmmBackward0>)

nn.Softmax
==========

The last linear layer of the neural network returns [logits]{.title-ref}
- raw values in \[-infty, infty\] - which are passed to the
[nn.Softmax](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html)
module. The logits are scaled to values \[0, 1\] representing the
model\'s predicted probabilities for each class. `dim` parameter
indicates the dimension along which the values must sum to 1.


In [14]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

In [15]:
pred_probab

tensor([[0.0730, 0.1048, 0.0926, 0.0855, 0.1215, 0.1065, 0.0936, 0.0973, 0.1159,
         0.1091],
        [0.0703, 0.0986, 0.0842, 0.0826, 0.1123, 0.1211, 0.0885, 0.0955, 0.1356,
         0.1115],
        [0.0747, 0.1030, 0.0942, 0.0913, 0.1120, 0.1096, 0.0889, 0.0958, 0.1209,
         0.1097]], grad_fn=<SoftmaxBackward0>)

Model Parameters
================

Many layers inside a neural network are *parameterized*, i.e. have
associated weights and biases that are optimized during training.
Subclassing `nn.Module` automatically tracks all fields defined inside
your model object, and makes all parameters accessible using your
model\'s `parameters()` or `named_parameters()` methods.

In this example, we iterate over each parameter, and print its size and
a preview of its values.


In [16]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0211,  0.0074, -0.0242,  ...,  0.0025,  0.0134,  0.0337],
        [-0.0352,  0.0129, -0.0246,  ..., -0.0337, -0.0353, -0.0267]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([0.0019, 0.0186], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[-0.0082,  0.0018, -0.0184,  ..., -0.0282, -0.0056,  0.0418],
        [ 0.0353,  0.0296,  0.0330,  ..., -0.0344, -0.0195, -0.0168]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | Si

In [17]:
bias = model.linear_relu_stack[0].bias
print(bias.shape)       # torch.Size([512])
print(bias[:10])        # 查看前10个元素
print(bias[-10:])       # 查看后10个元素


torch.Size([512])
tensor([ 0.0019,  0.0186,  0.0084, -0.0120,  0.0347, -0.0295,  0.0176,  0.0163,
         0.0258, -0.0211], grad_fn=<SliceBackward0>)
tensor([-0.0026, -0.0340,  0.0226, -0.0269, -0.0316, -0.0175,  0.0076,  0.0226,
        -0.0237, -0.0310], grad_fn=<SliceBackward0>)


------------------------------------------------------------------------


Further Reading
===============

-   [torch.nn API](https://pytorch.org/docs/stable/nn.html)
